In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\2025_Major Dhyan Chand National Stadium, Delhi - DPCC.xlsx",skiprows=16)

In [3]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,NH3,SO2,CO,Ozone,Benzene,Toluene,RH,WS,WD,BP,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,202.96,263.25,32.50,49.08,52.52,56.54,9.08,1.60,20.18,1.56,8.06,77.96,0.79,217.08,993.97,14.17,0.0,0.0
1,02-01-2025 00:00,03-01-2025 00:00,195.17,255.96,31.04,45.02,48.10,67.98,10.06,1.16,19.73,1.58,6.79,79.31,0.80,215.89,993.21,14.32,0.0,0.0
2,03-01-2025 00:00,04-01-2025 00:00,279.88,384.83,97.32,71.48,117.13,86.91,12.25,1.61,19.76,2.55,12.78,83.40,0.66,178.68,993.05,15.05,0.0,0.0
3,04-01-2025 00:00,05-01-2025 00:00,279.59,337.04,51.21,67.60,77.57,96.98,8.36,0.72,17.99,2.87,12.64,84.34,0.92,152.14,991.33,15.08,0.0,0.0
4,05-01-2025 00:00,06-01-2025 00:00,189.21,240.79,21.29,42.11,39.71,75.90,5.79,0.83,12.21,1.68,4.11,81.27,1.03,156.01,991.21,14.57,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,287.54,445.79,55.76,82.34,89.45,71.82,29.00,1.81,56.57,2.72,15.86,58.02,0.68,217.62,990.65,18.71,0.0,0.0
316,13-11-2025 00:00,14-11-2025 00:00,251.46,426.42,44.27,93.06,85.11,80.27,26.95,1.51,79.29,2.46,16.20,60.85,0.67,225.52,992.10,18.36,0.0,0.0
317,14-11-2025 00:00,15-11-2025 00:00,202.00,359.46,35.37,93.72,78.61,73.30,24.21,1.61,64.02,2.33,16.65,60.86,0.66,223.22,992.06,18.12,0.0,0.0
318,15-11-2025 00:00,16-11-2025 00:00,211.81,361.03,54.04,92.69,92.76,70.49,23.05,1.89,7.05,2.54,19.60,60.73,0.63,222.63,993.03,17.92,0.0,0.0


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (320, 20)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): []
Dropped rows (>70% NaN): 0
Missing values after imputation:
 From Date    0
To Date      0
PM2.5        0
PM10         0
NO           0
NO2          0
NOx          0
NH3          0
SO2          0
CO           0
Ozone        0
Benzene      0
Toluene      0
RH           0
WS           0
WD           0
BP           0
AT           0
RF           0
TOT-RF       0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:

# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (320, 20)
          From Date           To Date   PM2.5    PM10     NO    NO2     NOx  \
0  01-01-2025 00:00  02-01-2025 00:00  60.145  263.25  32.50  49.08   52.52   
1  02-01-2025 00:00  03-01-2025 00:00  60.145  255.96  31.04  45.02   48.10   
2  03-01-2025 00:00  04-01-2025 00:00  60.145  384.83  97.32  71.48  117.13   
3  04-01-2025 00:00  05-01-2025 00:00  60.145  337.04  51.21  67.60   77.57   
4  05-01-2025 00:00  06-01-2025 00:00  60.145  240.79  21.29  42.11   39.71   

     NH3    SO2    CO  Ozone  Benzene  Toluene     RH    WS      WD      BP  \
0  56.54   9.08  1.60  20.18     1.56     8.06  77.96  0.79  217.08  993.97   
1  67.98  10.06  1.16  19.73     1.58     6.79  79.31  0.80  215.89  993.21   
2  86.91  12.25  1.61  19.76     2.55    12.78  83.40  0.66  178.68  993.05   
3  63.28   8.36  0.72  17.99     2.87    12.64  84.34  0.92  152.14  991.33   
4  75.90   5.79  0.83  12.21     1.68     4.11  81.27  1.03  156.01  991.21   

      AT   RF  TOT-RF  
0  

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,NH3,SO2,CO,Ozone,Benzene,Toluene,RH,WS,WD,BP,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,-0.064323,1.386522,0.331724,0.032267,0.198245,-0.653179,0.026101,2.108571,-0.941660,-0.027375,-0.402084,1.336120,-0.922435,1.033543,1.794383,-2.266379,0.0,0.0
1,02-01-2025 00:00,03-01-2025 00:00,-0.064323,1.297203,0.273972,-0.161445,0.058528,0.332162,0.258177,0.835472,-0.962060,-0.008952,-0.631891,1.449530,-0.878859,0.997511,1.675700,-2.238856,0.0,0.0
2,03-01-2025 00:00,04-01-2025 00:00,-0.064323,2.876153,2.895766,1.101024,2.240591,1.962626,0.776796,2.137506,-0.960700,0.884556,0.451998,1.793119,-1.488914,-0.129172,1.650714,-2.104912,0.0,0.0
3,04-01-2025 00:00,05-01-2025 00:00,-0.064323,2.290617,1.071823,0.915900,0.990085,-0.072654,-0.144404,-0.437628,-1.040942,1.179322,0.426665,1.872086,-0.355956,-0.932778,1.382114,-2.099408,0.0,0.0
4,05-01-2025 00:00,06-01-2025 00:00,-0.064323,1.111336,-0.111702,-0.300288,-0.206683,1.014321,-0.753011,-0.119353,-1.302972,0.083162,-1.116836,1.614184,0.123373,-0.815598,1.363375,-2.192985,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,-0.064323,-0.120997,1.251805,1.619181,1.365616,0.662906,-0.151508,-0.003617,0.708043,1.041151,1.009323,-0.338982,-1.401763,1.049893,1.275924,-1.433359,0.0,0.0
316,13-11-2025 00:00,14-11-2025 00:00,-0.064323,-0.120997,0.797302,2.130657,1.228427,1.390715,-0.151508,1.848165,1.738031,0.801654,1.070846,-0.101242,-1.445338,1.289098,1.502359,-1.497578,0.0,0.0
317,14-11-2025 00:00,15-11-2025 00:00,-0.064323,2.565313,0.445251,2.162147,1.022960,0.790380,-0.151508,2.137506,1.045781,0.681905,1.152274,-0.100402,-1.488914,1.219456,1.496113,-1.541615,0.0,0.0
318,15-11-2025 00:00,16-11-2025 00:00,-0.064323,2.584549,1.183768,2.113004,1.470246,0.548352,-0.151508,-0.003617,-1.536895,0.875345,1.686075,-0.111323,-1.619639,1.201591,1.647590,-1.578312,0.0,0.0


In [10]:
df.to_excel('majordhyamChandStadium2025.xlsx', index=False)